<a href="https://colab.research.google.com/github/romanakki23/Flyrank_my_work/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/romanakki23/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Supervised Learning (Binary Classification & Learning-to-Rank)

For Lane 2, the problem is framed as a binary classification task to predict whether a given page is in a state of content decline (is_declining_label = 1), combined with a ranking function that converts predicted decline probabilities into a prioritized 0–100 score (refresh_score). This output allows content teams to consume a sorted queue rather than unorganized predictions.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/Proxy Definition: is_declining_label

Primary Target: Binary indicator derived from trend_direction == "down".

Proxy Nature: While the ideal long-term target would be future post-refresh organic traffic recovery over 30–90 days, historical performance logs make traffic trend direction (down vs. stable/up) a reliable observable proxy for pages experiencing exposure loss.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success Metric: Precision@K (specifically Precision@50) and ROC AUC

Precision@50: Measures what proportion of the top 50 flagged pages in the refresh queue are actually declining pages requiring intervention. This directly reflects content team capacity.

ROC AUC / PR AUC: Evaluates how effectively the model separates declining pages from stable or growing pages across all threshold cutoffs.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: A single pseudonymized content item (content_id / content_hash_id) aggregated over a 90-day observation window. One row represents one distinct URL's search performance, engagement, and metadata state.

In [3]:
import os
import pandas as pd

# Path resolution for Colab / local environments
possible_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
]

csv_path = None
for path in possible_paths:
    if os.path.exists(path):
        csv_path = path
        break

if csv_path is None:
    csv_path = "https://raw.githubusercontent.com/romanakki23/flyrank/main/data/raw/content_refresh_anonymized.csv"

# Load data and build explicit unit of analysis + target preview
df = pd.read_csv(csv_path)

# Filter for minimum exposure criteria
df_clean = df[(df["impressions_90d"] >= 100) & (df["content_age_days"] >= 90)].copy()

# Construct target variable
df_clean["is_declining_label"] = (df_clean["trend_direction"] == "down").astype(int)

# Display Unit of Analysis Dataframe
display_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
    "is_declining_label",
]
print(f"Dataset Shape (Rows = Units of Analysis): {df_clean.shape}")
df_clean[display_cols].head(10)

Dataset Shape (Rows = Units of Analysis): (22006, 45)


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,down,1
7,content_a63219c6e95a,client_19581e27de,1724,28,445,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,down,1
9,content_c27558df2b0c,client_19581e27de,1240,3,257,down,1
10,content_d8ee6cc6d642,client_19581e27de,20919,326,329,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML Beats Fixed Heuristics:
Static rules (e.g., "flag if age > 180 days and impressions drop 20%") fail because content performance is non-linear and multi-factorial. A page with high search volume, dropping CTR, and aging content requires different prioritization than a low-volume page that recently dropped.

A machine learning model learns subtle non-linear interactions across 20+ features simultaneously (volume, engagement rates, position tiers, freshness decay). In the baseline starter pipeline, fixed rules yield a Precision@50 of ~0.24, while learned classifiers (Random Forest) reach ~0.74 Precision@50—a ~3x improvement in reviewer efficiency.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.